In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings("ignore")

## MANOVA (다변량 분산분석)

## <문제 1>

한 교육 연구팀이 수업 방식(Traditional / Blended / Online) 과 성별(Male / Female) 이 학생의 수학 성취(math_score) 와 읽기 성취(reading_score) 에 미치는 영향을 조사하고자 합니다. 표본은 각 집단 조합당 30명씩이며, 두 성취도는 서로 상관을 가질 수 있습니다.

- teaching_method (범주형: Traditional, Blended, Online)

- gender (범주형) Male // Female

- math_score (연속형)

- reading_score (연속형)

- study_hours (참고용 연속형, 기본 MANOVA에는 사용 안 해도 됨)

In [3]:
file_path = "./data/manova.csv"
df = pd.read_csv(file_path)
df

,teaching_method,gender,math_score,reading_score,study_hours
0,Traditional,Male,83.772428,79.144917,5.979254
1,Traditional,Female,84.583224,75.015033,8.013109
2,Online,Female,74.012756,74.551889,5.731091
3,Blended,Male,66.968855,68.233970,3.741544
4,Online,Male,82.265137,82.282552,6.782187
...,...,...,...,...,...
175,Blended,Male,92.810662,88.847823,2.901920
176,Blended,Female,74.837869,76.685894,9.535417
177,Traditional,Male,80.341095,72.572685,13.382956
178,Blended,Female,83.018692,84.333647,5.535685


In [4]:
### 원본 데이터
df

,teaching_method,gender,math_score,reading_score,study_hours
0,Traditional,Male,83.772428,79.144917,5.979254
1,Traditional,Female,84.583224,75.015033,8.013109
2,Online,Female,74.012756,74.551889,5.731091
3,Blended,Male,66.968855,68.233970,3.741544
4,Online,Male,82.265137,82.282552,6.782187
...,...,...,...,...,...
175,Blended,Male,92.810662,88.847823,2.901920
176,Blended,Female,74.837869,76.685894,9.535417
177,Traditional,Male,80.341095,72.572685,13.382956
178,Blended,Female,83.018692,84.333647,5.535685


In [5]:
df["teaching_method"].value_counts()

Online         60
Blended        60
Traditional    60
Name: teaching_method, dtype: int64

In [6]:
### 그룹 나누기
cond = (df["teaching_method"] == "Online")
cond1 = (df["teaching_method"] == "Blended")
cond2 = (df["teaching_method"] == "Traditional")

gOnline = df.loc[cond, ["math_score", "reading_score"]].reset_index(drop=True).copy()
gBlended = df.loc[cond1, ["math_score", "reading_score"]].reset_index(drop=True).copy()
gTraditional = df.loc[cond2, ["math_score", "reading_score"]].reset_index(drop=True).copy()

In [7]:
### 다변량 정규성 검정 (Henze-Zirkler 검정)
import pingouin as pg

# H0 : 데이터가 다변량 정규성을 만족한다.
# H1 : 데이터가 다변량 정규성을 만족하지 않는다.

result, p = pg.multivariate_normality(gOnline, alpha=0.05)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.4699946771477544
귀무가설 채택


In [8]:
### 다변량 정규성 검정 (Henze-Zirkler 검정)
import pingouin as pg

# H0 : 데이터가 다변량 정규성을 만족한다.
# H1 : 데이터가 다변량 정규성을 만족하지 않는다.

result, p = pg.multivariate_normality(gBlended, alpha=0.05)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.30390172340853194
귀무가설 채택


In [9]:
### 다변량 정규성 검정 (Henze-Zirkler 검정)
import pingouin as pg

# H0 : 데이터가 다변량 정규성을 만족한다.
# H1 : 데이터가 다변량 정규성을 만족하지 않는다.

result, p = pg.multivariate_normality(gTraditional, alpha=0.05)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.5663269808447187
귀무가설 채택


In [11]:
# ### 공분산 행렬의 동질성 검정 (box-m test)
# import pingouin as pg

# result = pg.box_m(df,                                    ## 원본데이터
#                   dvs=["math_score", "reading_score"],   #  종속변수 컬럼명
#                   group="teaching_method"                #  독립변수 컬럼명 
#                   )
# result

In [12]:
### MANOVA 모델 적합
from statsmodels.multivariate.manova import MANOVA

# H0 : 수업방식별 수학성취와 읽기성취의 평균벡터는 같다.
# H1 : 적어도 하나의 수업방식의 수학성취와 읽기성취의 평균벡터가 다르다.

formula = "math_score + reading_score ~ C(teaching_method)"
model = MANOVA.from_formula(formula, data=df).mv_test()

model.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                   Multivariate linear model
================================================================
                                                                
----------------------------------------------------------------
       Intercept         Value  Num DF  Den DF   F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda  0.0175 2.0000 176.0000 4944.3224 0.0000
         Pillai's trace  0.9825 2.0000 176.0000 4944.3224 0.0000
 Hotelling-Lawley trace 56.1855 2.0000 176.0000 4944.3224 0.0000
    Roy's greatest root 56.1855 2.0000 176.0000 4944.3224 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
      C(teaching_method)   Value  Num DF  Den DF  F Value Pr > F
----------------------------------------------------------------
             Wilks' lambda 0.6442 4.0000 352.0000 21.6418 0.0000
            Pillai's trace 0.3745 4.0000 354.0000 20.3864 0.0000
    Hotelling-Lawley trace 0.5234 4.0000 210.1644 22.9868 0.0000
       Roy's greatest root 0.4606 2.0000 177.0000 40.7608 0.0000
================================================================

"""

In [13]:
print("""
귀무가설을 기각하므로 사후검정을 실시한다.
""")


귀무가설을 기각하므로 사후검정을 실시한다.



In [14]:
### 사후검정
from itertools import combinations

### 그룹을 2개씩 묶기
groups = df["teaching_method"].unique()
group_pairs = list(combinations(groups, 2))

for i in group_pairs:
    # 독립변수 값 2개씩 필터링 
    cond = df["teaching_method"].isin(i)
    df_temp = df.loc[cond, :].copy()
    
    # 필터링 데이터를 가지고 MANOVA 적합
    formula = "math_score + reading_score ~ C(teaching_method)"
    model = MANOVA.from_formula(formula, data=df_temp).mv_test()
    print(f"{i} 사후검정")
    print(model.summary())

('Traditional', 'Online') 사후검정
                   Multivariate linear model
                                                                
----------------------------------------------------------------
       Intercept         Value  Num DF  Den DF   F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda  0.0118 2.0000 117.0000 4911.3791 0.0000
         Pillai's trace  0.9882 2.0000 117.0000 4911.3791 0.0000
 Hotelling-Lawley trace 83.9552 2.0000 117.0000 4911.3791 0.0000
    Roy's greatest root 83.9552 2.0000 117.0000 4911.3791 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
      C(teaching_method)   Value  Num DF  Den DF  F Value Pr > F
----------------------------------------------------------------
             Wilks' lambda 0.5767 2.0000 117.0000 42.9331 0.0000
            Pi

## <문제 2>

한 스포츠과학 연구소에서는 서로 다른 운동프로그램이 신체 건강 지표에 미치는 효과를 성별에 따라 비교 분석하고자 한다. 연구진은 3가지 운동프로그램(유산소, 근력운동, 복합운동)에 참여한 남성과 여성 참가자들의 안정시 심박수와 체지방률을 동시에 측정하였다.

1. 운동프로그램은 신체 건강 지표(안정시심박수, 체지방률)에 유의한 영향을 미치는가?
2. 성별은 신체 건강 지표(안정시심박수, 체지방률)에 유의한 영향을 미치는가?
3. 운동프로그램과 성별 간의 상호작용 효과가 존재하는가?
4. 위의 효과들을 종합하여 어떤 결론을 내릴 수 있으며, 구체적인 그룹 간 차이는 무엇인가? (사후검정)

In [19]:
file_path = "./data/2way_manova.csv"

df = pd.read_csv(file_path)
df

,참가자ID,운동프로그램,성별,안정시심박수,체지방률
0,S001,유산소,남성,66.8,12.0
1,S002,유산소,남성,60.7,15.4
2,S003,유산소,남성,65.1,8.7
3,S004,유산소,남성,60.1,10.2
4,S005,유산소,남성,63.5,14.3
...,...,...,...,...,...
115,S116,복합운동,여성,64.2,18.1
116,S117,복합운동,여성,63.5,12.6
117,S118,복합운동,여성,68.8,16.4
118,S119,복합운동,여성,61.7,14.8


In [40]:
### 원본 데이터
df

,참가자ID,운동프로그램,성별,안정시심박수,체지방률
0,S001,유산소,남성,66.8,12.0
1,S002,유산소,남성,60.7,15.4
2,S003,유산소,남성,65.1,8.7
3,S004,유산소,남성,60.1,10.2
4,S005,유산소,남성,63.5,14.3
...,...,...,...,...,...
115,S116,복합운동,여성,64.2,18.1
116,S117,복합운동,여성,63.5,12.6
117,S118,복합운동,여성,68.8,16.4
118,S119,복합운동,여성,61.7,14.8


In [21]:
### 그룹 나누기
df["운동프로그램"].value_counts()

복합운동    40
근력운동    40
유산소     40
Name: 운동프로그램, dtype: int64

In [22]:
df["성별"].value_counts()

남성    60
여성    60
Name: 성별, dtype: int64

In [28]:
### 그룹 나누기
cond = (df["운동프로그램"] == "복합운동") & (df["성별"] == "남성")
cond1 = (df["운동프로그램"] == "복합운동") & (df["성별"] == "여성")
cond2 = (df["운동프로그램"] == "근력운동") & (df["성별"] == "남성")
cond3 = (df["운동프로그램"] == "근력운동") & (df["성별"] == "여성")
cond4 = (df["운동프로그램"] == "유산소") & (df["성별"] == "남성")
cond5 = (df["운동프로그램"] == "유산소") & (df["성별"] == "여성")

gMix_M = df.loc[cond, ["안정시심박수", "체지방률"]].reset_index(drop=True).copy()
gMix_W = df.loc[cond1, ["안정시심박수", "체지방률"]].reset_index(drop=True).copy()
gMuscle_M = df.loc[cond2, ["안정시심박수", "체지방률"]].reset_index(drop=True).copy()
gMuscle_W = df.loc[cond3, ["안정시심박수", "체지방률"]].reset_index(drop=True).copy()
gYu_M = df.loc[cond4, ["안정시심박수", "체지방률"]].reset_index(drop=True).copy()
gYu_W = df.loc[cond5, ["안정시심박수", "체지방률"]].reset_index(drop=True).copy()

In [30]:
### 다변량 정규성 검정 (Henze-Zirkler Test) => gMix_M 그룹
import pingouin as pg

pg.multivariate_normality(gMix_M)

(True, 0.8063143748510048)

In [31]:
### 다변량 정규성 검정 (Henze-Zirkler Test) => gMix_W 그룹
import pingouin as pg

pg.multivariate_normality(gMix_W)

(True, 0.09796944976545513)

In [32]:
### 다변량 정규성 검정 (Henze-Zirkler Test) => gMuscle_M 그룹
import pingouin as pg

pg.multivariate_normality(gMuscle_M)

(True, 0.7106804652639906)

In [33]:
### 다변량 정규성 검정 (Henze-Zirkler Test) => gMuscle_W 그룹
import pingouin as pg

pg.multivariate_normality(gMuscle_W)

(True, 0.10452028608230635)

In [34]:
### 다변량 정규성 검정 (Henze-Zirkler Test) => gYu_M 그룹
import pingouin as pg

pg.multivariate_normality(gYu_M)

(True, 0.09558932063403591)

In [35]:
### 다변량 정규성 검정 (Henze-Zirkler Test) => gYu_W 그룹
import pingouin as pg

pg.multivariate_normality(gYu_W)

(True, 0.4677938128126112)

In [ ]:
### 공분산 행렬의 동질성 검정
import pingouin as pg

result = pg.box_m(df,                                 ## 원본 데이터
                  dvs=["안정시심박수", "체지방률"],   # 종속변수 컬럼명
                  group=["운동프로그램", "성별"]      # 독립변수 컬럼명
                  )
result

In [54]:
df

,참가자ID,운동프로그램,성별,안정시심박수,체지방률
0,S001,유산소,남성,66.8,12.0
1,S002,유산소,남성,60.7,15.4
2,S003,유산소,남성,65.1,8.7
3,S004,유산소,남성,60.1,10.2
4,S005,유산소,남성,63.5,14.3
...,...,...,...,...,...
115,S116,복합운동,여성,64.2,18.1
116,S117,복합운동,여성,63.5,12.6
117,S118,복합운동,여성,68.8,16.4
118,S119,복합운동,여성,61.7,14.8


In [42]:
### MANOVA 적합하기
from statsmodels.multivariate.manova import MANOVA

formula = "안정시심박수 + 체지방률 ~ C(운동프로그램)*C(성별)"
model = MANOVA.from_formula(formula, data=df).mv_test()

model.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                   Multivariate linear model
================================================================
                                                                
----------------------------------------------------------------
       Intercept         Value  Num DF  Den DF   F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda  0.0118 2.0000 113.0000 4719.4752 0.0000
         Pillai's trace  0.9882 2.0000 113.0000 4719.4752 0.0000
 Hotelling-Lawley trace 83.5305 2.0000 113.0000 4719.4752 0.0000
    Roy's greatest root 83.5305 2.0000 113.0000 4719.4752 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
          C(운동프로그램)        Value  Num DF  Den DF  F Value Pr > F
----------------------------------------------------------------
             Wilks' lambda 0.5594 4.0000 226.0000 19.0410 0.0000
            Pillai's trace 0.4984 4.0000 228.0000 18.9183 0.0000
    Hotelling-Lawley trace 0.6843 4.0000 134.5668 19.2751 0.0000
       Roy's greatest root 0.4593 2.0000 114.0000 26.1829 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
            C(성별)          Value  Num DF  Den DF  F Value Pr > F
----------------------------------------------------------------
             Wilks' lambda 0.5218 2.0000 113.0000 51.7860 0.0000
            Pillai's trace 0.4782 2.0000 113.0000 51.7860 0.0000
    Hotelling-Lawley trace 0.9166 2.0000 113.0000 51.7860 0.0000
       Roy's greatest root 0.9166 2.0000 113.0000 51.7860 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
       C(운동프로그램):C(성별)     Value  Num DF  Den DF  F Value Pr > F
----------------------------------------------------------------
             Wilks' lambda 0.9890 4.0000 226.0000  0.3137 0.8686
            Pillai's trace 0.0110 4.0000 228.0000  0.3158 0.8672
    Hotelling-Lawley trace 0.0111 4.0000 134.5668  0.3135 0.8686
       Roy's greatest root 0.0105 2.0000 114.0000  0.5976 0.5518
================================================================

"""

In [43]:
print("""
[가설 설정]

<주효과1 검정>
H0 : 운동 프로그램별 안정시심박수와 체지방률의 평균 벡터가 같다.
H1 : 적어도 하나의 운동 프로그램의 안정시심박수와 체지방률의 평균 벡터가 다르다.

<주효과2 검정>
H0 : 성별간 안정시심박수와 체지방률의 평균 벡터가 같다.
H1 : 성별간 안정시심박수오 체지방률의 평균 벡터가 다르다.

<상호작용 효과 검정>
H0 : 운동프로그램과 성별간 상호작용 효과가 없다.
H1 : 운동프로그램과 성별간 상호작용 효과가 있다.
""")


[가설 설정]

<주효과1 검정>
H0 : 운동 프로그램별 안정시심박수와 체지방률의 평균 벡터가 같다.
H1 : 적어도 하나의 운동 프로그램의 안정시심박수와 체지방률의 평균 벡터가 다르다.

<주효과2 검정>
H0 : 성별간 안정시심박수와 체지방률의 평균 벡터가 같다.
H1 : 성별간 안정시심박수오 체지방률의 평균 벡터가 다르다.

<상호작용 효과 검정>
H0 : 운동프로그램과 성별간 상호작용 효과가 없다.
H1 : 운동프로그램과 성별간 상호작용 효과가 있다.



In [44]:
print("""
주효과1과 주효과2는 귀무가설 기각이며
상호작용효과는 귀무가설 채택으로 상호작용 효과가 없음을 확인하였다.
""")


주효과1과 주효과2는 귀무가설 기각이며
상호작용효과는 귀무가설 채택으로 상호작용 효과가 없음을 확인하였다.



In [52]:
### 주효과1 사후검정
from itertools import combinations

### 그룹을 2개씩 묶기
groups = df["운동프로그램"].unique()
group_pairs = list(combinations(groups, 2))

for i in group_pairs:
    # 독립변수 값 2개씩 필터링
    cond = df["운동프로그램"].isin(i)
    df_temp = df.loc[cond, :].copy()
    
    # 필터링 데이터를 가지고 MANOVA 적합
    formula = "안정시심박수 + 체지방률 ~ C(운동프로그램)"
    model = MANOVA.from_formula(formula, data=df_temp).mv_test()
    print(f"{i} 사후검정")
    print(model.summary())

('유산소', '근력운동') 사후검정
                   Multivariate linear model
                                                                
----------------------------------------------------------------
       Intercept         Value   Num DF  Den DF  F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda   0.0052 2.0000 77.0000 7385.9809 0.0000
         Pillai's trace   0.9948 2.0000 77.0000 7385.9809 0.0000
 Hotelling-Lawley trace 191.8437 2.0000 77.0000 7385.9809 0.0000
    Roy's greatest root 191.8437 2.0000 77.0000 7385.9809 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
           C(운동프로그램)        Value  Num DF  Den DF F Value Pr > F
----------------------------------------------------------------
              Wilks' lambda 0.6597 2.0000 77.0000 19.8613 0.0000
             Pillai's tr

In [53]:
### 주효과2 사후검정
from itertools import combinations

### 그룹을 2개씩 묶기
groups = df["성별"].unique()
group_pairs = list(combinations(groups, 2))

for i in group_pairs:
    # 독립변수 값 2개씩 필터링
    cond = df["성별"].isin(i)
    df_temp = df.loc[cond, :].copy()
    
    # 필터링 데이터를 가지고 MANOVA 적합
    formula = "안정시심박수 + 체지방률 ~ C(성별)"
    model = MANOVA.from_formula(formula, data=df_temp).mv_test()
    print(f"{i} 사후검정")
    print(model.summary())

('남성', '여성') 사후검정
                    Multivariate linear model
                                                                 
-----------------------------------------------------------------
       Intercept         Value   Num DF  Den DF   F Value  Pr > F
-----------------------------------------------------------------
          Wilks' lambda   0.0080 2.0000 117.0000 7232.0415 0.0000
         Pillai's trace   0.9920 2.0000 117.0000 7232.0415 0.0000
 Hotelling-Lawley trace 123.6246 2.0000 117.0000 7232.0415 0.0000
    Roy's greatest root 123.6246 2.0000 117.0000 7232.0415 0.0000
-----------------------------------------------------------------
                                                                 
-----------------------------------------------------------------
             C(성별)          Value  Num DF  Den DF  F Value Pr > F
-----------------------------------------------------------------
              Wilks' lambda 0.3700 2.0000 117.0000 99.6109 0.0000
            